In [103]:
import mlflow
from mlflow.models.signature import infer_signature

import keras

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

from hyperopt import STATUS_OK, Trials, fmin, hp, tpe, space_eval

import pandas as pd
import numpy as np


In [125]:
mlflow.set_tracking_uri("http://localhost:5000")



In [93]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-red.csv",
    sep=";"
)

data.head()



,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [127]:
X = data.drop("quality", axis=1)
y = data["quality"]




In [128]:
y = y.astype(float)


In [129]:
# scaler = StandardScaler()
# Xs = scaler.fit_transform(X)


In [143]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 


X_train.shape, X_test.shape, y_train.shape, y_test.shape


((1279, 11), (320, 11), (1279,), (320,))

In [144]:
X_train = X_train.to_numpy() if hasattr(X_train, "to_numpy") else X_train
X_test  = X_test.to_numpy()  if hasattr(X_test, "to_numpy")  else X_test

X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")
y_train = np.asarray(y_train).astype("float32")
y_test  = np.asarray(y_test).astype("float32")


In [154]:

def train_model(params, X_train, y_train, X_test, y_test, return_model=False):    
    
    model = keras.Sequential()
    norm = keras.layers.Normalization()
    norm.adapt(X_train)

    
    model.add(keras.Input(shape=(X_train.shape[1],)))
    
   
    model.add(norm)
    
    for _ in range(params["n_layers"]):
        model.add(
            keras.layers.Dense(
                params["units"],
                activation=params["activation"]
            )
        )
        if params["dropout"] > 0:
            model.add(keras.layers.Dropout(params["dropout"]))
    
    model.add(keras.layers.Dense(1))
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["lr"]),
        loss="mean_squared_error",
        metrics=["mse", "mae"]
    )
    
    with mlflow.start_run(nested=True):
        model.fit(
            X_train, y_train,
            epochs=params["epochs"],
            validation_data=(X_test, y_test),
            batch_size=params["batch_size"],
            verbose=0
        )
        
        eval_results = model.evaluate(X_test, y_test, batch_size=params["batch_size"])

        mlflow.log_metrics({
            "mse": eval_results[1],
            "mae": eval_results[2]
        })
        
        mlflow.log_params(params)
        
        return_data = {
            "loss": eval_results[0],
            "mae": eval_results[2],
            "status": STATUS_OK
        }
    
        if return_model:
            return_data["model"] = model

        return return_data


In [155]:
def objective(params):
    return train_model(params, X_train, y_train, X_test, y_test)


In [156]:
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-2)),
    "epochs": hp.choice("epochs", [3, 5, 10]),
    "units": hp.choice("units", [16, 32, 64]),
    "batch_size": hp.choice("batch_size", [8, 16, 32]),
    "activation": hp.choice("activation", ["relu", "tanh", "sigmoid"]),
    # "optimizer": hp.choice("optimizer", ["adam", "sgd", "rmsprop"]),
    "dropout": hp.uniform("dropout", 0.0, 0.4),
    "n_layers": hp.choice("n_layers", [1, 2, 3])
}



In [157]:
mlflow.set_experiment("wine-quality")

with mlflow.start_run():
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=10,
        trials=trials
    )
    best_params = space_eval(space, best)
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})




 1/40 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.7357 - mae: 1.3731 - mse: 2.7357
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.1924 - mae: 1.4724 - mse: 3.1924 

🏃 View run crawling-cat-272 at: http://localhost:5000/#/experiments/2/runs/a32562fd21bd42d8aab9f4ceeb482200

🧪 View experiment at: http://localhost:5000/#/experiments/2

 1/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.4886 - mae: 0.5480 - mse: 0.4886
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5060 - mae: 0.5558 - mse: 0.5060 

🏃 View run merciful-shrike-803 at: http://localhost:5000/#/experiments/2/runs/d704242172234909b8fb976f27a7e5ea

🧪 View experiment at: http://localhost:5000/#/experiments/2                    

 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 32.1062 - mae: 5.5834 - mse: 32.1062
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 32.7547 - mae: 5.6577 - mse: 32.7547 

🏃 View run gregarious-wolf-630 at: http://localhost:5000/#/experiments/2/runs/e52aa086810d4b6da8696b2c36066edf

🧪 View experiment at

In [149]:
best_params


{'activation': 'relu',
 'batch_size': 16,
 'dropout': 0.01220331153104013,
 'epochs': 10,
 'lr': 0.00980456723710552,
 'n_layers': 3,
 'units': 64}

In [159]:
with mlflow.start_run(run_name="best_model"):
    best_model = train_model(best_params, X_train, y_train, X_test, y_test, return_model=True)
    
    mlflow.log_params(best_params)
    
    mlflow.log_metrics({
        "mse": best_model["loss"],
        "mae": best_model["mae"]
    })
    
    mlflow.keras.log_model(
        best_model['model'],
        "best_model",
        # input_example=X_test[:5]
    )



40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3954 - mae: 0.5017 - mse: 0.3954 


2026/02/02 20:42:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run secretive-moth-313 at: http://localhost:5000/#/experiments/2/runs/6838d455843349f2920f74b683766041
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/02/02 20:42:07 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/02/02 20:42:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run best_model at: http://localhost:5000/#/experiments/2/runs/dc479b9a8de14a6f86a4bf7312488322
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [ ]:
best_model


In [122]:
best_model['model'].summary()


Model: "sequential_58"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_126 (Dense)               │ (None, 16)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_127 (Dense)               │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,445 (5.65 KB)

 Trainable params: 481 (1.88 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 964 (3.77 KB)